# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Model Evaluation for Regression

Evaluate regression models with comprehensive metrics and cross-validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Step 1: Load and Prepare Data

In [ ]:
# Load california housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 2: Regression Metrics

In [ ]:
# Train a baseline model
model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

# Mean Absolute Error: average absolute error
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.4f}")
print(f"  → On average, predictions are off by {mae:.2f} units")

# Mean Squared Error: average squared error
mse = mean_squared_error(y_test, y_pred)
print(f"\nMSE: {mse:.4f}")

# Root Mean Squared Error: back to original units, penalizes large errors
rmse = np.sqrt(mse)
print(f"\nRMSE: {rmse:.4f}")
print(f"  → Square root of MSE, penalizes large errors more")

# R-squared: proportion of variance explained (0 to 1, higher is better)
r2 = r2_score(y_test, y_pred)
print(f"\nR²: {r2:.4f}")
print(f"  → Model explains {r2*100:.1f}% of variance")
print(f"  → 1.0 = perfect, 0.0 = no better than mean, <0 = worse than mean")

## Step 3: Cross-Validation

Single train/test split can be biased. K-fold cross-validation gives more reliable estimate.

In [ ]:
# 5-fold cross-validation
model = LinearRegression()
cv_scores = cross_val_score(
    model, X_train_scaled, y_train,
    cv=5, scoring='r2'
)

print(f"Cross-validation R² scores: {cv_scores}")
print(f"Mean R²: {cv_scores.mean():.4f}")
print(f"Std:     {cv_scores.std():.4f}")
print(f"\nInterpretation: Model's R² is {cv_scores.mean():.2f} ± {cv_scores.std():.2f}")

## Step 4: Hyperparameter Tuning with GridSearchCV

In [ ]:
# Ridge regression with different alpha (regularization strength)
param_grid = {
    'alpha': [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV tries each combination and picks the best (without peeking at test set)
grid_search = GridSearchCV(
    Ridge(), param_grid,
    cv=5, scoring='r2', n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print(f"Best alpha: {grid_search.best_params_['alpha']}")
print(f"Best cross-validation R²: {grid_search.best_score_:.4f}")

# Evaluate on test set
best_model = grid_search.best_estimator_
test_r2 = best_model.score(X_test_scaled, y_test)
print(f"\nTest set R²: {test_r2:.4f}")

## Step 5: Compare Results

In [ ]:
# Summary table
print("=" * 50)
print("MODEL COMPARISON")
print("=" * 50)

# Linear regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_r2 = lr.score(X_test_scaled, y_test)
print(f"\nLinear Regression:")
print(f"  Test R²: {lr_r2:.4f}")

# Best ridge from grid search
print(f"\nBest Ridge (alpha={grid_search.best_params_['alpha']}):")
print(f"  Test R²: {test_r2:.4f}")

print(f"\nImprovement: {(test_r2 - lr_r2):.4f}")